# Colab Validation Run — Phases 1–6 on Real WOMD Data

Phases 1–6 are committed and tested, but **only against synthetic scenarios built as numpy
arrays** — two vehicles, all 91 timesteps valid, no crowding, no occlusion. Nothing in
Phases 4, 5, or 6 has ever touched a real WOMD shard. `export_shard_geometry` has never
executed at all.

This is a **validation run, not a demo**. The goal is diagnostics: find out where the
synthetic fixtures lied. Every cell below prints its findings even when the news is boring —
"0 interior gaps" is a result, not a skip.

The first run of this notebook measured two Phase 3 bugs on the real shard. Both are now
**fixed** (commits "Fix PET to evaluate both crossing orders" and "Rank Pass 1 on
SDC-restricted TTC/PET instead of all pairs"), and this version re-runs against the same
shard to confirm the fixes hold on real data:

1. **Pass 1 scoring was SDC-agnostic.** `score_scenario` now takes `sdc_index` and ranks on
   TTC/PET restricted to pairs involving the SDC; the old scene-wide values are retained as
   `min_ttc_all_pairs` / `min_pet_all_pairs` diagnostics. Cell 7 shows the saturation
   before/after; cell 7c shows how the ranking moved.
2. **PET evaluated only one crossing order.** `compute_pet_pair` now returns
   `max(enter_b - exit_a, enter_a - exit_b)`, so a negative PET means genuine simultaneous
   occupancy. Cell 7d is a standing regression check on that.

Neither was a correctness bug in Phase 4 — every shipped stress-test result verified its
collision against exact SAT geometry regardless of ranking. This was a prioritisation fix.

> ⚠️ **If you edit any file under `src/` and re-run a cell, Colab does NOT see the change.**
> Python caches the imported module. The fix is **Runtime → Restart session**, then re-run
> from the top — not `del sys.modules[...]`, which is fragile and easy to get subtly wrong
> with submodules.

This notebook writes only to a session-local Postgres database and reads the repo + the
shard. **It never writes back to the repo.**

## 1. Protobuf environment variable

Must be set **before any import**, in its own cell, before anything (including this
notebook's own later cells) imports `google.protobuf` transitively. Setting it after import
has no effect — the C++ implementation is already loaded.

In [ ]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
print("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION =",
      os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'])

## 2. Installs

`--no-deps` on the Waymo package: it pins an old TensorFlow we do not need (`loader.py`
parses TFRecord with `struct`; only `scenario_pb2` from the Waymo package is used at
runtime). `torch` is deliberately **not installed** — `autograd_optimizer.py` imports it at
module level, but `batch_scorer._stress_one` only imports that module when
`use_autograd=True`, and this notebook runs DE-only, so it is never needed.

Every resolved version is printed so a future failure has a known-good baseline to diff
against.

In [ ]:
!pip install waymo-open-dataset-tf-2-11-0 --no-deps --quiet
!pip install shapely scipy psycopg2-binary --quiet

import importlib.metadata as ilm

for pkg in ["waymo-open-dataset-tf-2-11-0", "shapely", "scipy", "psycopg2-binary",
            "numpy", "protobuf"]:
    try:
        print(f"{pkg:35s} {ilm.version(pkg)}")
    except ilm.PackageNotFoundError:
        print(f"{pkg:35s} NOT INSTALLED")

import sys
print("\npython", sys.version)

## 3. Repo + Drive

Clones the repo if absent, otherwise pulls `main`. Mounts Drive and **asserts the shard file
exists before anything else runs** — failing here, loudly, beats discovering a missing file
300 lines into a batch pass.

`REPO_URL`, `SHARD_PATH` and the run-size knobs (`MAX_SCENARIOS`, `TOP_N`, `DIAG_N`) are the
only things you should need to change to rerun this at a different scale.

In [ ]:
# ── config (the only cell you should need to edit for a different run) ──────
REPO_URL = "https://github.com/AviShrivastava1/av_stress_tester.git"
REPO_DIR = "/content/av_stress_tester"
SHARD_PATH = ("/content/drive/MyDrive/waymo_data/"
              "uncompressed_scenario_training_training.tfrecord-00000-of-01000")

MAX_SCENARIOS = 100   # Pass 1 batch size
TOP_N = 5             # how many scenarios get the Phase 4 stress test (Pass 2)
DIAG_N = 50           # sample size for the 7c/7d rank-correlation + PET experiments
DE_KWARGS = dict(popsize=10, maxiter=60, tol=1e-2, seed=1)

PG_USER = "avi"
PG_PASSWORD = "avi"   # a real (non-empty) password — see cell 4 for why '' cannot work
PG_DB = "av_stress"

In [ ]:
import os, subprocess, sys

if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already present — pulling latest main")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)
else:
    print(f"Cloning {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("sys.path[0] =", sys.path[0])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

assert os.path.exists(SHARD_PATH), (
    f"Shard not found at {SHARD_PATH}. Check the Drive mount and the path before "
    f"running anything else — every later cell depends on this file."
)
size_gb = os.path.getsize(SHARD_PATH) / (1024 ** 3)
print(f"Shard found: {SHARD_PATH}")
print(f"Size: {size_gb:.2f} GB")

## 4. Postgres + PostGIS

Colab has no Postgres preinstalled. This installs it fresh, creates role `avi` and database
`av_stress`, enables PostGIS, and runs both schema-init functions. **Colab's disk resets
between sessions, so this database is per-session scratch — that is fine for a validation
run**; nothing here needs to persist.

`PGUSER`/`PGDATABASE`/`PGPASSWORD` are exported into the environment now because
`src/api/config.py` builds its `Settings()` singleton **at import time** — cell 14 must not
be the first thing to read these.

**The role needs a real password, not `''`.** An empty string sets Postgres's password
field to NULL (Postgres itself warns and clears it), which cannot satisfy password
authentication — and TCP connections to `localhost` use `scram-sha-256` by default on
Ubuntu, so `db.get_connection()` would fail immediately with
`fe_sendauth: no password supplied`. `PG_PASSWORD` from the config cell is used both when
creating the role and when exporting `PGPASSWORD`, so the two can never drift apart.

**The PostGIS package name is discovered at runtime, not hardcoded.** Colab's base Ubuntu
image drifts between sessions, and a hardcoded `postgresql-14-postgis-3` breaks outright the
day the image ships Postgres 15 or 16. Installing plain `postgresql` first and reading
`/usr/lib/postgresql/` back tells us which major version actually landed, so the PostGIS
package name is always right for the image running right now. If the specific
`-postgis-3` package genuinely doesn't exist for that major version, this fails loudly with
the actual apt-cache search output rather than raising anywhere from a wrong package name.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y postgresql postgresql-contrib > /dev/null

# Discover the Postgres major version actually installed, rather than hardcoding one —
# Colab's base image drifts between sessions and a fixed version number breaks silently.
import os as _os
pg_versions = sorted(_os.listdir('/usr/lib/postgresql'))
assert pg_versions, "postgresql install did not create /usr/lib/postgresql — apt failed silently"
pg_major = pg_versions[-1]
postgis_pkg = f"postgresql-{pg_major}-postgis-3"
print(f"Detected Postgres major version: {pg_major}  ->  installing {postgis_pkg}")

import subprocess as sp
result = sp.run(["apt-get", "install", "-y", "-qq", postgis_pkg], capture_output=True, text=True)
if result.returncode != 0:
    print(f"FAILED to install {postgis_pkg}. apt-cache search results for 'postgis':")
    search = sp.run(["apt-cache", "search", "postgis"], capture_output=True, text=True)
    print(search.stdout)
    raise RuntimeError(
        f"{postgis_pkg} is not installable on this image. See the candidates printed "
        f"above and adjust postgis_pkg by hand for this session."
    )

!service postgresql start

sp.run(["sudo", "-u", "postgres", "psql", "-c",
        f"CREATE ROLE {PG_USER} WITH SUPERUSER LOGIN PASSWORD '{PG_PASSWORD}';"],
       capture_output=True)
sp.run(["sudo", "-u", "postgres", "createdb", "-O", PG_USER, PG_DB],
       capture_output=True)
sp.run(["sudo", "-u", "postgres", "psql", "-d", PG_DB, "-c",
        "CREATE EXTENSION IF NOT EXISTS postgis;"], check=True)

os.environ['PGHOST'] = 'localhost'
os.environ['PGPORT'] = '5432'
os.environ['PGUSER'] = PG_USER
os.environ['PGPASSWORD'] = PG_PASSWORD
os.environ['PGDATABASE'] = PG_DB

print(f"Postgres running, role={PG_USER}, database={PG_DB}")

In [ ]:
from src.scoring import db
from src.scoring.export_geometry import init_geometry_schema

conn = db.get_connection()
db.init_schema(conn)
init_geometry_schema(conn)

with conn.cursor() as cur:
    cur.execute("SELECT PostGIS_Version();")
    print("PostGIS_Version():", cur.fetchone()[0])
conn.commit()
print("Schema initialized: scenario_scores, scenario_agents, perturbed_paths")

## 5. Smoke test — one scenario

The cheapest possible check that Phase 1 still works against this specific shard, before
any batch work. If this cell fails, nothing downstream will work either.

In [ ]:
from src.data.loader import ShardLoader
from src.data.parser import ScenarioParser

loader = ShardLoader(SHARD_PATH)
raw = next(iter(loader))
parser = ScenarioParser(raw)

states = parser.get_agent_states()
validity = parser.get_agent_validity()
types = parser.get_agent_types()
sdc_idx = parser.get_sdc_index()

print("scenario_id:", parser.get_scenario_id())
print("states.shape:", states.shape)
print("validity.shape:", validity.shape)
print("sdc_index:", sdc_idx)

import numpy as np
uniq, counts = np.unique(types, return_counts=True)
type_names = {0: "UNSET", 1: "VEHICLE", 2: "PEDESTRIAN", 3: "CYCLIST", 4: "OTHER"}
print("agent type counts:")
for u, c in zip(uniq, counts):
    print(f"  {type_names.get(int(u), f'unknown({u})'):10s} {c}")

## 6. Timing probe, then Pass 1 (`score_shard`)

PET rebuilds swept-polygon unions inside `compute_pet_pair`, so its cost grows with agent
density — something the synthetic fixtures (2 agents) never exercised. This probes 3 real
scenarios first and **projects the cost for `MAX_SCENARIOS`**, so the run can be resized
before committing to it rather than discovered 20 minutes in.

In [ ]:
import time
from src.scoring.batch_scorer import _score_one

probe_records = []
t0 = time.time()
for i, raw in enumerate(ShardLoader(SHARD_PATH)):
    if i >= 3:
        break
    p = ScenarioParser(raw)
    rec = _score_one(p.get_agent_states(), p.get_agent_validity(),
                     p.get_scenario_id(), p.get_sdc_index())
    probe_records.append(rec)
probe_elapsed = time.time() - t0
per_scenario = probe_elapsed / len(probe_records)

print(f"Probe: {len(probe_records)} scenarios in {probe_elapsed:.2f}s "
      f"({per_scenario:.2f}s/scenario)")
print(f"Projected for MAX_SCENARIOS={MAX_SCENARIOS}: "
      f"{per_scenario * MAX_SCENARIOS:.1f}s (~{per_scenario * MAX_SCENARIOS / 60:.1f} min)")
print(f"Projected for a full 1000-shard dataset at this rate: "
      f"~{per_scenario * MAX_SCENARIOS * 1000 / 3600:.1f} hours "
      f"(scaling this probe's per-scenario cost, not a measured full run)")

In [ ]:
from src.scoring.batch_scorer import score_shard

t0 = time.time()
records, errors = score_shard(SHARD_PATH, max_scenarios=MAX_SCENARIOS,
                              pet_max_pairs=50, progress_every=25, verbose=True)
pass1_elapsed = time.time() - t0
print(f"\nPass 1 done: {len(records)} scored, {len(errors)} errored, "
      f"{pass1_elapsed:.1f}s total")

## 7. Pass 1 diagnostics

The most valuable cell in the batch section. Covers: error isolation (the first real
exercise of the per-scenario try/except in `score_shard`), the fragility distribution, TTC/PET
saturation rates, agent-count distribution, and validity-gap statistics across every parsed
track — including **interior gaps**, the case that breaks any "vertex index equals timestep"
assumption (see cell 13).

In [ ]:
import numpy as np
from collections import Counter

# ── error isolation ──────────────────────────────────────────────────────────
print("=" * 70)
print("ERROR ISOLATION")
print("=" * 70)
print(f"scored: {len(records)}   errored: {len(errors)}")
if errors:
    exc_types = Counter(e['error'].split(':')[0] for e in errors)
    print("distinct exception types:")
    for exc, n in exc_types.most_common():
        print(f"  {exc:30s} {n}")
else:
    print("0 errors — clean pass on this sample")

# ── fragility distribution ───────────────────────────────────────────────────
frag = np.array([r['fragility_score'] for r in records])
print()
print("=" * 70)
print("FRAGILITY SCORE DISTRIBUTION")
print("=" * 70)
print(f"min={frag.min():.4f}  max={frag.max():.4f}  mean={frag.mean():.4f}")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{p:<3d} {np.percentile(frag, p):.4f}")

# ── TTC / PET saturation — SDC-restricted (production) vs all-pairs ────────────
# Import the REAL constants rather than hardcoding literals — a diagnostic that
# reimplements a threshold with a literal will silently drift from the function
# it is describing. (An earlier draft of this notebook had exactly that bug: it
# compared `== 0.0` when the real clamp is `<= TTC_FLOOR`, and `== 999.0` when the
# real sentinel test is `< TTC_INFINITY`. Both under-counted saturation.)
from src.danger.danger_score import TTC_FLOOR, PET_FLOOR
from src.danger.ttc_engine import TTC_INFINITY
from src.danger.pet_engine import PET_INFINITY

# After the Phase 3 rework, score_scenario returns BOTH: min_ttc/min_pet are the
# SDC-restricted values it now ranks on, and min_ttc_all_pairs/min_pet_all_pairs
# are the old scene-wide numbers, kept as a diagnostic. The direct before/after
# for finding 1 is comparing the saturation of the two, on the same scenarios.
ttc_sdc = np.array([r['min_ttc'] for r in records])
pet_sdc = np.array([r['min_pet'] for r in records])
ttc_all = np.array([r['min_ttc_all_pairs'] for r in records])
pet_all = np.array([r['min_pet_all_pairs'] for r in records])

def _sat(arr, floor):
    # (exact-zero mask, floor-capped mask, sentinel mask). Floor-capped is what
    # actually pins the danger signal at its max: compute_danger_score clamps via
    # max(x, floor), so anything <= floor behaves identically to an exact 0.0.
    is_ttc = (floor == TTC_FLOOR)
    sentinel = arr >= (TTC_INFINITY if is_ttc else PET_INFINITY)
    return (arr == 0.0), (arr <= floor), sentinel

ttc_sdc_zero, ttc_sdc_cap, ttc_sdc_sent = _sat(ttc_sdc, TTC_FLOOR)
ttc_all_zero, ttc_all_cap, ttc_all_sent = _sat(ttc_all, TTC_FLOOR)
pet_sdc_zero, pet_sdc_cap, pet_sdc_sent = _sat(pet_sdc, PET_FLOOR)
pet_all_zero, pet_all_cap, pet_all_sent = _sat(pet_all, PET_FLOOR)

pet_sdc_negative = (pet_sdc < 0.0)   # finding 5, production signal, full sample

print()
print("=" * 70)
print("TTC / PET SATURATION (finding 1) — SDC-restricted vs all-pairs")
print("=" * 70)
print(f"                              SDC-restricted (ranked on)   all-pairs (old)")
print(f"min_ttc <= TTC_FLOOR ({TTC_FLOOR}):   "
      f"{ttc_sdc_cap.sum():4d}/{len(ttc_sdc)} ({100*ttc_sdc_cap.mean():5.1f}%)        "
      f"{ttc_all_cap.sum():4d}/{len(ttc_all)} ({100*ttc_all_cap.mean():5.1f}%)")
print(f"min_pet <= PET_FLOOR ({PET_FLOOR}):   "
      f"{pet_sdc_cap.sum():4d}/{len(pet_sdc)} ({100*pet_sdc_cap.mean():5.1f}%)        "
      f"{pet_all_cap.sum():4d}/{len(pet_all)} ({100*pet_all_cap.mean():5.1f}%)")
print(f"min_ttc >= INFINITY sentinel:  "
      f"{ttc_sdc_sent.sum():4d}/{len(ttc_sdc)} ({100*ttc_sdc_sent.mean():5.1f}%)        "
      f"{ttc_all_sent.sum():4d}/{len(ttc_all)} ({100*ttc_all_sent.mean():5.1f}%)")
print()
if ttc_all_cap.mean() > 0.5 and ttc_sdc_cap.mean() < ttc_all_cap.mean() - 0.1:
    print(f"-> FINDING 1 CONFIRMED FIXED: all-pairs TTC saturated "
          f"{100*ttc_all_cap.mean():.0f}% of scenarios (uninformative); "
          f"SDC-restricted saturates {100*ttc_sdc_cap.mean():.0f}%, leaving "
          f"{(~ttc_sdc_cap).sum()} scenarios with a discriminating TTC signal.")
elif ttc_all_cap.mean() <= 0.5:
    print(f"-> all-pairs TTC was NOT heavily saturated on this sample "
          f"({100*ttc_all_cap.mean():.0f}%) — smaller shard slice than the "
          f"100/100 validation run, or a quieter shard region.")
else:
    print(f"-> SDC-restricted saturation ({100*ttc_sdc_cap.mean():.0f}%) is not "
          f"materially below all-pairs ({100*ttc_all_cap.mean():.0f}%). Investigate "
          f"before trusting the new ranking on this data.")
print()
print(f"min_pet < 0.0, SDC-restricted (finding 5 — now genuine overlaps only): "
      f"{pet_sdc_negative.sum()}/{len(pet_sdc)} ({100*pet_sdc_negative.mean():.1f}%)")

# non-saturated fragility percentiles: exclude scenarios whose PRODUCTION signal
# (SDC-restricted TTC) is floor-capped. frag is already the SDC-restricted score.
non_sat = frag[~ttc_sdc_cap]
print()
print(f"fragility percentiles over the NON-saturated subset (SDC-restricted TTC "
      f"above floor, n={len(non_sat)}):")
if len(non_sat):
    for p in [10, 25, 50, 75, 90, 95, 99]:
        print(f"  p{p:<3d} {np.percentile(non_sat, p):.4f}")
else:
    print("  (every scenario's SDC-restricted TTC saturated — unexpected post-rework, "
          "see the summary cell)")

# names cell 48 still references
ttc_floor_capped = ttc_sdc_cap
pet_negative = pet_sdc_negative

# ── agent count distribution ──────────────────────────────────────────────────
n_agents = np.array([r['n_agents'] for r in records])
print()
print("=" * 70)
print("AGENT COUNT (n_agents) DISTRIBUTION")
print("=" * 70)
print(f"min={n_agents.min()}  median={int(np.median(n_agents))}  max={n_agents.max()}")

# ── PET pair-selection diagnostics (finding 4) — index arithmetic only, no geometry ──
print()
print("=" * 70)
print("PET PAIR-SELECTION DIAGNOSTICS (finding 4)")
print("=" * 70)
print("NOTE post-rework: the production ranking now uses compute_min_pet_sdc, which")
print("is SDC-anchored and UNCAPPED, so this degeneracy no longer affects the ranking.")
print("It still describes min_pet_all_pairs, the retained scene-density diagnostic.")


def _pet_pairs_checked(n_agents_i, max_pairs=50):
    """Replicate compute_min_pet_scenario's (i, j) selection with i<j, no geometry."""
    pairs = []
    for i in range(n_agents_i):
        for j in range(i + 1, n_agents_i):
            if len(pairs) >= max_pairs:
                return pairs
            pairs.append((i, j))
    return pairs


ge_51 = 0
sdc_never_checked = 0
anchor_counts = []
for r in records:
    n = r['n_agents']
    if n >= 51:
        ge_51 += 1
    pairs = _pet_pairs_checked(n)
    anchors = len(set(i for i, j in pairs))
    anchor_counts.append(anchors)
    # sdc_idx is not stored on the record; re-derive is not available here without
    # re-parsing. This diagnostic instead reports the STRUCTURAL degeneracy (anchor
    # count) directly — see cell 7d for the SDC-specific version, computed on the
    # cached sample where sdc_idx is available.

print(f"scenarios with n_agents >= 51 (cap exhausted inside i=0): "
      f"{ge_51}/{len(records)} ({100*ge_51/len(records):.1f}%)")
print(f"distinct 'i' anchors reached per scenario: "
      f"min={min(anchor_counts)}  median={int(np.median(anchor_counts))}  "
      f"max={max(anchor_counts)}")
print("(anchors == 1 means every checked PET pair shares agent 0 in that scenario;")
print(" the SDC-specific version of this — whether the SDC is ever checked at all —")
print(" is computed in cell 7d against the cached sample, where sdc_idx is available.)")

# ── validity-gap statistics ───────────────────────────────────────────────────
print()
print("=" * 70)
print("VALIDITY-GAP STATISTICS")
print("=" * 70)
print("(recomputed per scenario in this pass — see cell 7b for the cached-sample version")
print(" used by cells 7c/7d; this section re-walks the shard once more, which is the")
print(" price of getting gap stats on the FULL MAX_SCENARIOS sample rather than DIAG_N)")

fully_valid = prefix_suffix_only = interior_gap = total_tracks = 0
t0 = time.time()
wanted_ids = set(r['scenario_id'] for r in records)
for raw in ShardLoader(SHARD_PATH):
    if not wanted_ids:
        break
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    if sid not in wanted_ids:
        continue
    wanted_ids.discard(sid)
    v = p.get_agent_validity()
    for i in range(v.shape[0]):
        total_tracks += 1
        row = v[i]
        if row.all():
            fully_valid += 1
            continue
        valid_idx = np.where(row)[0]
        if len(valid_idx) == 0:
            continue
        # contiguous prefix/suffix only: the valid indices form one unbroken run
        span = valid_idx[-1] - valid_idx[0] + 1
        if span == len(valid_idx):
            prefix_suffix_only += 1
        else:
            interior_gap += 1
gap_scan_elapsed = time.time() - t0

print(f"total agent tracks examined: {total_tracks}  ({gap_scan_elapsed:.1f}s)")
print(f"  fully valid (all 91 timesteps):        {fully_valid} "
      f"({100*fully_valid/max(total_tracks,1):.1f}%)")
print(f"  contiguous run (no interior gap):       {prefix_suffix_only} "
      f"({100*prefix_suffix_only/max(total_tracks,1):.1f}%)")
print(f"  INTERIOR GAP (breaks index==timestep):   {interior_gap} "
      f"({100*interior_gap/max(total_tracks,1):.1f}%)")

# ── timing projection ──────────────────────────────────────────────────────
print()
print("=" * 70)
print("WALL-CLOCK PROJECTION")
print("=" * 70)
mean_sec = np.mean([r['score_seconds'] for r in records])
print(f"mean seconds/scenario (this pass): {mean_sec:.3f}s")
print(f"projected for a full 1000-shard dataset at ~{mean_sec:.3f}s/scenario "
      f"and this shard's scenario count as a stand-in: see the cell-6 probe for the "
      f"MAX_SCENARIOS-scale projection; this line uses the Pass-1-measured rate instead "
      f"of the 3-scenario probe rate, so compare the two for probe reliability.")

## 7b. Diagnostic sample cache

Cells 7c and 7d both need parsed arrays (`states`, `validity`, `types`, `sdc_idx`) for the
same sample of scenarios, and the shard lives on Drive-mounted storage where a second and
third sequential read would be the dominant cost of running this section. So this cell
parses `DIAG_N` scenarios once and holds them in memory.

Cheap to hold: a 100-agent scenario is `91 × 100 × 7 × 4 bytes ≈ 255 KB`, so `DIAG_N=50`
scenarios is on the order of tens of MB — printed below rather than assumed.

In [ ]:
import sys as _sys

diag_cache = []
for raw in ShardLoader(SHARD_PATH):
    if len(diag_cache) >= DIAG_N:
        break
    p = ScenarioParser(raw)
    diag_cache.append({
        'scenario_id': p.get_scenario_id(),
        'states': p.get_agent_states(),
        'validity': p.get_agent_validity(),
        'types': p.get_agent_types(),
        'sdc_idx': p.get_sdc_index(),
    })

resident_bytes = sum(
    d['states'].nbytes + d['validity'].nbytes + d['types'].nbytes
    for d in diag_cache
)
print(f"Cached {len(diag_cache)} scenarios for cells 7c/7d "
      f"({resident_bytes / 1024**2:.1f} MB resident)")

## 7c. Rank-correlation experiment — confirms the finding-1 fix

Computes each cached scenario's danger profile **twice**, changing only the pair set, and
both paths are now real production code (before the rework, the SDC-only path was
hand-reimplemented here — that reimplementation-drift risk is why the old 7d needed a
validity gate):

- **all-pairs (old ranking)** — `compute_danger_score(compute_min_ttc_scenario(...),
  compute_min_pet_scenario(...))`, the scene-wide functions Phase 3 used to rank on
- **SDC-restricted (new ranking)** — `score_scenario(...)['fragility_score']`, exactly what
  Pass 1 now produces

Both go through `rank_scenarios`. This is the concrete before/after: which scenarios the
old ranking sent to Phase 4 that the new one does not, and vice versa.

**Interpretation:** a low Spearman rho with small top-N overlap is the fix working — the two
signals genuinely rank scenarios differently, and the validation run measured rho = -0.0152
with 2/5 overlap on this data. A rho near 1.0 would mean the rework changed nothing and the
motivation was wrong.

Cost note: `score_scenario` runs BOTH pair sets internally (SDC-restricted for the score,
all-pairs for the diagnostic), so the all-pairs recompute below is redundant work — kept
explicit so the comparison reads clearly. `DIAG_N` is tuned from the cell-6 probe.

In [ ]:
from src.danger.ttc_engine import compute_min_ttc_scenario
from src.danger.pet_engine import compute_min_pet_scenario
from src.danger.danger_score import compute_danger_score, score_scenario
from src.scoring.ranker import rank_scenarios

t0 = time.time()
all_pairs_records = []
sdc_prod_records = []

for d in diag_cache:
    states, validity, sdc_idx = d['states'], d['validity'], d['sdc_idx']

    # ---- old ranking: scene-wide all-pairs ----
    min_ttc_all = compute_min_ttc_scenario(states, validity)
    min_pet_all = compute_min_pet_scenario(states, validity, max_pairs=50)
    frag_all = compute_danger_score(min_ttc_all, min_pet_all)
    all_pairs_records.append({'scenario_id': d['scenario_id'], 'fragility_score': frag_all,
                              'min_ttc': min_ttc_all, 'min_pet': min_pet_all})

    # ---- new ranking: exactly what Pass 1 now writes ----
    rec = score_scenario(states, validity, d['scenario_id'], sdc_idx)
    sdc_prod_records.append({'scenario_id': d['scenario_id'],
                             'fragility_score': rec['fragility_score'],
                             'min_ttc': rec['min_ttc'], 'min_pet': rec['min_pet']})

elapsed_7c = time.time() - t0
print(f"7c compute: {elapsed_7c:.1f}s for {len(diag_cache)} scenarios "
      f"({elapsed_7c/len(diag_cache):.2f}s/scenario)")

In [ ]:
from scipy.stats import spearmanr

ranked_all = rank_scenarios(all_pairs_records)
ranked_sdc = rank_scenarios(sdc_prod_records)

order_all = [r['scenario_id'] for r in ranked_all]
order_sdc = [r['scenario_id'] for r in ranked_sdc]

rank_all = {sid: i for i, sid in enumerate(order_all)}
rank_sdc = {sid: i for i, sid in enumerate(order_sdc)}
common = list(rank_all.keys())

rho, pval = spearmanr([rank_all[s] for s in common], [rank_sdc[s] for s in common])

top10_all = set(order_all[:10])
top10_sdc = set(order_sdc[:10])
topN_all = set(order_all[:TOP_N])
topN_sdc = set(order_sdc[:TOP_N])

frag_all_by = {r['scenario_id']: r['fragility_score'] for r in ranked_all}
frag_sdc_by = {r['scenario_id']: r['fragility_score'] for r in ranked_sdc}

print("=" * 70)
print("RANK-CORRELATION: old all-pairs ranking vs new SDC-restricted ranking")
print("=" * 70)
print(f"Spearman rho: {rho:.4f}  (p={pval:.4g})   [validation run measured -0.0152]")
print(f"top-10 set overlap: {len(top10_all & top10_sdc)}/10")
print(f"top-{TOP_N} set overlap (the set Phase 4 actually runs on): "
      f"{len(topN_all & topN_sdc)}/{TOP_N}")

print()
print(f"scenarios in the OLD top-{TOP_N} that the NEW ranking drops:")
for sid in topN_all - topN_sdc:
    print(f"  {sid}  old_frag={frag_all_by[sid]:.3f} (rank {rank_all[sid]})  ->  "
          f"new_frag={frag_sdc_by[sid]:.4f} (rank {rank_sdc[sid]})")
print(f"scenarios the NEW top-{TOP_N} adds:")
for sid in topN_sdc - topN_all:
    print(f"  {sid}  new_frag={frag_sdc_by[sid]:.3f} (rank {rank_sdc[sid]})  <-  "
          f"old_frag={frag_all_by[sid]:.4f} (rank {rank_all[sid]})")

print()
if abs(rho) < 0.3 and len(topN_all & topN_sdc) < TOP_N:
    print("VERDICT: the two rankings are effectively unrelated — the rework materially")
    print("changed which scenarios reach Phase 4, consistent with the validation run.")
else:
    print(f"VERDICT: rho={rho:.3f}, top-{TOP_N} overlap {len(topN_all & topN_sdc)}/{TOP_N}.")
    print("Larger overlap than the validation run — check whether this shard slice is")
    print("less crowded, so all-pairs was not saturating and the two signals agree.")

## 7d. PET sign-semantics regression — confirms the finding-5 fix

Before the rework this cell measured how many negative PETs were spurious. That is done —
`compute_pet_pair` now evaluates `max(enter_b - exit_a, enter_a - exit_b)`, so a negative
result can only mean the two agents' conflict-zone occupancy intervals genuinely overlap.

This cell is now a **standing regression check** on that guarantee, run against real data:
for every SDC pair in the cached sample it calls production `compute_pet_pair`, and for
each result that is negative it independently recomputes the occupancy intervals with
`pet_engine._occupancy_interval` (the same helper the function uses) and asserts they
actually overlap (`enter_a <= exit_b and enter_b <= exit_a`). A negative PET whose intervals
do **not** overlap would be a regression of finding 5.

No reimplementation, so no validity gate: `_occupancy_interval` and `get_path_polygon` are
imported from the module under test.

In [ ]:
from src.danger.pet_engine import (
    compute_pet_pair, get_path_polygon, _occupancy_interval, PET_INFINITY,
)

n_sdc_pairs = 0
n_negative = 0
n_zero = 0
n_positive = 0
n_infinity = 0
regressions = []   # negative PET whose intervals do NOT overlap — must stay empty

for d in diag_cache:
    states, validity, sdc_idx = d['states'], d['validity'], d['sdc_idx']
    N = states.shape[0]
    sdc_path = get_path_polygon(states, sdc_idx, validity)
    if sdc_path is None:
        continue
    for j in range(N):
        if j == sdc_idx:
            continue
        pet = compute_pet_pair(states, validity, sdc_idx, j, path_a=sdc_path)
        n_sdc_pairs += 1
        if pet >= PET_INFINITY:
            n_infinity += 1
            continue
        if pet > 0:
            n_positive += 1
            continue
        if pet == 0:
            n_zero += 1
            continue
        # pet < 0 — verify the intervals genuinely overlap
        n_negative += 1
        zone = sdc_path.intersection(get_path_polygon(states, j, validity))
        ea, xa = _occupancy_interval(states, validity, sdc_idx, zone)
        eb, xb = _occupancy_interval(states, validity, j, zone)
        intervals_overlap = (ea != -1 and eb != -1 and ea <= xb and eb <= xa)
        if not intervals_overlap:
            regressions.append((d['scenario_id'], sdc_idx, j, pet, (ea, xa), (eb, xb)))

print("=" * 70)
print("PET SIGN-SEMANTICS REGRESSION (finding 5)")
print("=" * 70)
print(f"SDC pairs evaluated:            {n_sdc_pairs}")
print(f"  PET_INFINITY (no shared zone / no crossing): {n_infinity}")
print(f"  positive (sequenced crossing, safe):         {n_positive}")
print(f"  exactly zero:                                {n_zero}")
print(f"  negative (genuine simultaneous occupancy):   {n_negative}")
print()
if regressions:
    print(f"*** REGRESSION: {len(regressions)} negative PET(s) with NON-overlapping intervals.")
    print("*** compute_pet_pair is returning a negative value that is not a genuine overlap —")
    print("*** finding 5 has regressed. Details:")
    for sid, a, b, pet, ia, ib in regressions[:10]:
        print(f"    {sid} pair=({a},{b}) pet={pet:.2f} a_interval={ia} b_interval={ib}")
    raise AssertionError(f"{len(regressions)} PET sign-semantics regressions — see above")
else:
    print("OK — every negative PET corresponds to genuinely overlapping occupancy intervals.")
    print("Finding 5's corrected semantics hold on real data.")

## 8. Rank + persist

Ranks the real Pass 1 output, prints the top 10, and persists via `upsert_scores`. Then
**upserts the same records a second time** and asserts the row count is unchanged —
idempotency verified against real data, not three synthetic rows.

In [ ]:
ranked = rank_scenarios(records)
print(f"{'rank':<6}{'scenario_id':<40}{'fragility':<12}{'min_ttc':<10}{'min_pet':<10}{'n_agents'}")
for r in ranked[:10]:
    print(f"{r['rank']:<6}{r['scenario_id']:<40}{r['fragility_score']:<12.4f}"
          f"{r['min_ttc']:<10.3f}{r['min_pet']:<10.3f}{r['n_agents']}")

In [ ]:
n1 = db.upsert_scores(conn, records)
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM scenario_scores")
    count_after_first = cur.fetchone()[0]

n2 = db.upsert_scores(conn, records)
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM scenario_scores")
    count_after_second = cur.fetchone()[0]

print(f"first upsert:  wrote {n1} rows, table now has {count_after_first}")
print(f"second upsert: wrote {n2} rows, table now has {count_after_second}")
assert count_after_first == count_after_second, (
    "IDEMPOTENCY VIOLATION: row count changed on a repeat upsert of identical records."
)
print("Idempotency confirmed on real data.")

## 9. Pass 2 — `stress_test_scenarios`

Runs the Phase 4 optimizer on the top `TOP_N` scenarios by fragility, with a small DE budget
so this stays fast enough to iterate on. Timed for the Pass-2 cost projection.

In [ ]:
from src.scoring.ranker import top_n_ids
from src.scoring.batch_scorer import stress_test_scenarios

ids_to_test = top_n_ids(records, TOP_N)
print("stress-testing:", ids_to_test)

t0 = time.time()
stress_results = stress_test_scenarios(SHARD_PATH, ids_to_test, de_kwargs=DE_KWARGS,
                                       verbose=True)
pass2_elapsed = time.time() - t0
print(f"\nPass 2 done: {pass2_elapsed:.1f}s for {len(ids_to_test)} scenarios "
      f"({pass2_elapsed/len(ids_to_test):.1f}s/scenario)")

## 10. Pass 2 diagnostics

Per-scenario table plus aggregates: how many hit `no_challenger`/`error`, how many
challengers were non-vehicles (the first real exercise of the linear-model path and of the
DE-only branch, since autograd is vehicle-only), and which perturbation component dominated
each solution — measured in `space.weights`-normalized units so speed (m/s) and heading
(rad) are comparable.

In [ ]:
type_names = {0: "UNSET", 1: "VEHICLE", 2: "PEDESTRIAN", 3: "CYCLIST", 4: "OTHER"}
component_names = ["dv0/dvx0", "dtheta0/dvy0", "da_bias/dax_bias", "ddelta_bias/day_bias"]

# re-parse just these scenarios to get types + build a PerturbationSpace for the weights
from src.optimization.perturbation_space import PerturbationSpace

challenger_types = {}
dominant_component = {}
for raw in ShardLoader(SHARD_PATH):
    if not ids_to_test:
        break
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    if sid not in stress_results:
        continue
    r = stress_results[sid]
    if r.get('status') != 'ok' or r.get('target_idx') is None:
        continue
    types_arr = p.get_agent_types()
    tgt = r['target_idx']
    challenger_types[sid] = int(types_arr[tgt])

    space = PerturbationSpace(p.get_agent_states(), p.get_agent_validity(), types_arr,
                              p.get_sdc_index(), tgt)
    weighted = np.abs(np.asarray(r['delta']) * space.weights)
    dominant_component[sid] = component_names[int(np.argmax(weighted))]

print(f"{'scenario_id':<40}{'status':<14}{'collision':<11}{'||delta||':<12}"
      f"{'t_hit':<8}{'target_idx':<12}{'challenger_type':<16}{'dominant'}")
n_no_challenger = n_error = n_collision = n_safe = n_non_vehicle = 0
for sid, r in stress_results.items():
    status = r.get('status')
    if status == 'no_challenger':
        n_no_challenger += 1
    elif status == 'error':
        n_error += 1
    collided = r.get('collision', False)
    if status == 'ok':
        if collided:
            n_collision += 1
        else:
            n_safe += 1
    ttype = challenger_types.get(sid)
    ttype_name = type_names.get(ttype, "?") if ttype is not None else "-"
    if ttype is not None and ttype != 1:
        n_non_vehicle += 1
    print(f"{sid:<40}{status:<14}{str(collided):<11}"
          f"{r.get('min_perturbation', float('nan')):<12.4f}"
          f"{r.get('collision_timestep', -1):<8}{r.get('target_idx', -1):<12}"
          f"{ttype_name:<16}{dominant_component.get(sid, '-')}")

print()
print("=" * 70)
print("PASS 2 AGGREGATES")
print("=" * 70)
print(f"no_challenger: {n_no_challenger}   error: {n_error}")
print(f"collision found: {n_collision}   robustly safe: {n_safe}")
print(f"non-vehicle challengers (types 0/2/3/4, first real linear-model exercise): "
      f"{n_non_vehicle}/{len(challenger_types)}")
type4_or_0 = sum(1 for t in challenger_types.values() if t in (0, 4))
print(f"  of which TYPE_UNSET(0) or TYPE_OTHER(4) specifically: {type4_or_0}")
print(f"seconds/scenario (from cell 9): {pass2_elapsed/max(len(ids_to_test),1):.1f}")
print(f"implied cost of a top-50 stress test: "
      f"{pass2_elapsed/max(len(ids_to_test),1)*50/60:.1f} min")

## 11. `update_stress_results`, then re-read

Writes the Phase 4 columns, then re-reads via `db.fetch_top` and confirms the new columns
landed and the Pass-1 columns (`fragility_score`, `min_ttc`, `min_pet`) were **untouched**.

In [ ]:
n_updated = db.update_stress_results(conn, stress_results)
print(f"updated {n_updated} rows")

top_rows = db.fetch_top(conn, n=TOP_N)
for row in top_rows:
    print(f"{row['scenario_id']:<40} stress_tested_at={row['stress_tested_at']} "
          f"min_perturbation={row['min_perturbation']} "
          f"fragility_score={row['fragility_score']:.4f}")

for row in top_rows:
    orig = next(r for r in records if r['scenario_id'] == row['scenario_id'])
    assert abs(row['fragility_score'] - orig['fragility_score']) < 1e-9, (
        f"Pass-1 fragility_score for {row['scenario_id']} changed after a Pass-2 write "
        f"— update_stress_results must not touch Pass-1 columns."
    )
    # the Phase 3 rework's additive columns must round-trip too
    for col in ('min_ttc', 'min_pet', 'min_ttc_all_pairs', 'min_pet_all_pairs'):
        assert col in row, f"{col} missing from fetch_top row — schema ALTER did not apply"
        if orig.get(col) is not None:
            assert abs(row[col] - orig[col]) < 1e-9, (
                f"{col} for {row['scenario_id']} does not match what Pass 1 wrote"
            )
print("\nConfirmed: Phase 4 columns landed; Pass-1 columns (incl. the new "
      "SDC-restricted and all-pairs diagnostic columns) untouched and round-tripped.")

## 12. Pass 3 — `export_shard_geometry`

**This code has never executed before this cell, on any data, synthetic or real.** It
shipped verified only by import. Prints the full summary dict and every error verbatim.

In [ ]:
from src.scoring.export_geometry import export_shard_geometry

t0 = time.time()
summary = export_shard_geometry(conn, SHARD_PATH, ids_to_test,
                                stress_results=stress_results, verbose=True)
pass3_elapsed = time.time() - t0

print()
print("=" * 70)
print("PASS 3 SUMMARY (export_shard_geometry — first-ever execution)")
print("=" * 70)
print(f"exported:           {summary['exported']}")
print(f"agents_written:     {summary['agents_written']}")
print(f"agents_skipped:     {summary['agents_skipped']}")
print(f"perturbed_written:  {summary['perturbed_written']}")
print(f"errors:             {len(summary['errors'])}")
for e in summary['errors']:
    print(f"  {e}")
print(f"elapsed: {pass3_elapsed:.1f}s")

## 13. Geometry verification — the most important assertion in this notebook

Re-parses the tested scenarios directly and compares the database against ground truth
computed independently in Python.

**The M-array comparison is the key assertion.** With every timestep valid (as in every
synthetic fixture used until now), `ST_M` values equal `0..N-1`, indistinguishable from
plain vertex indices — which is exactly why the original (buggy) exporter design could pass
every prior test. On real data with occlusion, an agent's M sequence should **jump** at a
gap. This cell proves the exporter wrote true timesteps, not vertex positions, by comparing
element-for-element against `np.where(validity[i])[0]`.

In [ ]:
def dump_points_m(conn, table, scenario_id, agent_idx=None):
    with conn.cursor() as cur:
        if table == 'scenario_agents':
            cur.execute("""
                SELECT agent_idx, ST_NPoints(path), headings,
                       ARRAY(SELECT ST_M(dp.geom) FROM ST_DumpPoints(path) dp
                             ORDER BY dp.path)
                FROM scenario_agents WHERE scenario_id = %s AND agent_idx = %s
            """, (scenario_id, agent_idx))
        else:
            cur.execute("""
                SELECT target_idx, ST_NPoints(path), headings,
                       ARRAY(SELECT ST_M(dp.geom) FROM ST_DumpPoints(path) dp
                             ORDER BY dp.path)
                FROM perturbed_paths WHERE scenario_id = %s
            """, (scenario_id,))
        return cur.fetchone()

In [ ]:
n_checked = 0
n_exact_match = 0
gap_example_found = False
skip_reconciled = 0

for raw in ShardLoader(SHARD_PATH):
    if not ids_to_test:
        break
    p = ScenarioParser(raw)
    sid = p.get_scenario_id()
    if sid not in ids_to_test:
        continue
    validity = p.get_agent_validity()
    N = validity.shape[0]

    with conn.cursor() as cur:
        cur.execute("SELECT agent_idx FROM scenario_agents WHERE scenario_id = %s", (sid,))
        exported_idxs = set(row[0] for row in cur.fetchall())

    for i in range(N):
        true_valid_ts = np.where(validity[i])[0]
        if len(true_valid_ts) < 2:
            skip_reconciled += int(i not in exported_idxs)
            continue
        if i not in exported_idxs:
            print(f"MISMATCH: agent {i} in {sid} has {len(true_valid_ts)} valid "
                  f"timesteps but was not exported")
            continue

        row = dump_points_m(conn, 'scenario_agents', sid, i)
        _, npoints, headings, m_values = row
        n_checked += 1

        assert npoints == len(true_valid_ts), (
            f"{sid} agent {i}: ST_NPoints={npoints} != {len(true_valid_ts)} valid timesteps"
        )
        m_array = np.array(m_values, dtype=int)
        if np.array_equal(m_array, true_valid_ts):
            n_exact_match += 1
        else:
            print(f"MISMATCH: {sid} agent {i}: M values {m_array[:10]}... != "
                  f"true valid timesteps {true_valid_ts[:10]}...")

        assert len(headings) == npoints, (
            f"{sid} agent {i}: len(headings)={len(headings)} != ST_NPoints={npoints}"
        )

        # is this a genuine interior-gap agent? report the first one found, verbatim.
        if not gap_example_found and len(true_valid_ts) >= 2:
            span = true_valid_ts[-1] - true_valid_ts[0] + 1
            if span != len(true_valid_ts):
                jump_pos = np.where(np.diff(true_valid_ts) > 1)[0][0]
                print(f"\nINTERIOR GAP EXAMPLE — {sid} agent {i}:")
                print(f"  M sequence around the jump: "
                      f"...{m_array[max(0,jump_pos-2):jump_pos+3]}...")
                print(f"  (jumps from {m_array[jump_pos]} to {m_array[jump_pos+1]}, "
                      f"skipping {m_array[jump_pos+1]-m_array[jump_pos]-1} invalid timesteps)")
                gap_example_found = True

print(f"\nagents checked: {n_checked}   exact M-array matches: {n_exact_match}")
assert n_exact_match == n_checked, "Some agents' M arrays did not match true valid timesteps."
print("CONFIRMED: M ordinate equals true valid-timestep indices, element-for-element.")

if not gap_example_found:
    print("\nNo interior-gap agent found among the tested scenarios' agents — reported")
    print("explicitly rather than silently: this sample happened not to contain one.")

print(f"\nagents with <2 valid timesteps, correctly skipped (not crashed on): "
      f"{skip_reconciled}")
print(f"reconciles against Pass 3's agents_skipped={summary['agents_skipped']} "
      f"(this loop only covers the {len(ids_to_test)} Pass-3 scenarios, so equality is "
      f"expected only if agents_skipped was computed over exactly this set)")

## 14. API check — the full round trip on real data

Starts uvicorn in a daemon thread (modern uvicorn skips installing signal handlers off the
main thread, so `Server.run()` works here) and polls `/health` until ready. `PGUSER` /
`PGPASSWORD` / `PGDATABASE` were exported back in cell 4 — before this is the first import
of `src.api.main`, since `Settings()` builds itself at import time.

The closing assertion is the point of this cell: confirm the `timesteps` array the HTTP
endpoint returns for a real scenario matches the M values read directly from Postgres in
cell 13 — proving the chain shard → exporter → PostGIS → HTTP holds together on real data.

In [ ]:
import threading
import uvicorn
import httpx

from src.api.main import app

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()

base = "http://127.0.0.1:8000"
deadline = time.time() + 30
ready = False
last_error = None
while time.time() < deadline:
    try:
        r = httpx.get(f"{base}/health", timeout=1.0)
        if r.status_code == 200:
            ready = True
            break
        last_error = f"HTTP {r.status_code}: {r.text[:200]}"
    except httpx.HTTPError as e:
        # Expected while uvicorn is still binding its socket — connection refused
        # is the normal state for the first second or so. Not silently swallowed:
        # the last one seen is reported below if the deadline expires without success.
        last_error = f"{type(e).__name__}: {e}"
    time.sleep(0.5)

assert ready, (
    f"uvicorn did not become ready within 30s. Last error: {last_error}. "
    f"Check PGUSER/PGPASSWORD/PGDATABASE were exported in cell 4 before this import."
)
print("Server ready.")

In [ ]:
print("GET /health"); print(httpx.get(f"{base}/health").json())
print()
print("GET /stats"); print(httpx.get(f"{base}/stats").json())
print()
print("GET /scenarios?limit=3")
print(httpx.get(f"{base}/scenarios", params={"limit": 3}).json())

stress_tested_sid = next(sid for sid, r in stress_results.items()
                         if r.get('status') == 'ok' and r.get('collision'))
print(f"\nGET /scenarios/{stress_tested_sid}/trajectories")
traj = httpx.get(f"{base}/scenarios/{stress_tested_sid}/trajectories").json()
print(f"  {len(traj['agents'])} agents; first agent timesteps[:10] = "
      f"{traj['agents'][0]['timesteps'][:10]}")

print(f"\nGET /scenarios/{stress_tested_sid}/perturbed")
pert = httpx.get(f"{base}/scenarios/{stress_tested_sid}/perturbed").json()
print(f"  delta={pert['delta']}  collision_timestep={pert['collision_timestep']}")

In [ ]:
# close the loop: HTTP timesteps must equal the M values read directly from Postgres
target_idx_api = pert['target_idx']
http_agent = next(a for a in traj['agents'] if a['agent_idx'] == target_idx_api)
http_timesteps = np.array(http_agent['timesteps'])

row = dump_points_m(conn, 'scenario_agents', stress_tested_sid, target_idx_api)
_, npoints_db, _, m_values_db = row
db_m = np.array(m_values_db, dtype=float)

assert np.array_equal(http_timesteps, db_m), (
    "HTTP /trajectories timesteps do not match the M values read directly from Postgres — "
    "the shard-to-HTTP round trip is broken somewhere between cells 12 and 14."
)
print("CONFIRMED: HTTP timesteps == Postgres M values. Full round trip closed on real data.")

server.should_exit = True

## 15. Summary

Self-contained — safe to copy out of the notebook on its own. Leads with the two numbers
that confirm the Phase 3 rework did what it was meant to.

In [ ]:
print("=" * 70)
print("COLAB VALIDATION RUN — SUMMARY  (post Phase 3 rework)")
print("=" * 70)
print()
print("-- Did the rework work? --")
print(f"[Finding 1] old all-pairs ranking vs new SDC-restricted ranking: "
      f"Spearman rho={rho:.4f}, top-{TOP_N} overlap={len(topN_all & topN_sdc)}/{TOP_N}")
print(f"            (low rho / small overlap = the rework changed the ranking, as intended;")
print(f"             validation baseline was rho=-0.0152, 2/5 overlap)")
print(f"[Finding 1] TTC floor-saturation: SDC-restricted {100*ttc_sdc_cap.mean():.1f}%  "
      f"vs all-pairs {100*ttc_all_cap.mean():.1f}%")
print(f"[Finding 5] negative SDC min_pet, all confirmed genuine overlaps: "
      f"{n_negative} pair(s), 0 sign-semantics regressions")
print()
print("-- Pass 1 --")
print(f"scenarios scored: {len(records)}   errored: {len(errors)}")
print(f"fragility range: [{frag.min():.4f}, {frag.max():.4f}]  mean={frag.mean():.4f}")
print(f"SDC-restricted min_ttc floor-capped: {100*ttc_sdc_cap.mean():.1f}%   "
      f"(all-pairs diagnostic: {100*ttc_all_cap.mean():.1f}%)")
print(f"SDC-restricted min_pet floor-capped: {100*pet_sdc_cap.mean():.1f}%   "
      f"negative: {100*pet_sdc_negative.mean():.1f}%")
print(f"agent tracks with an interior validity gap: {interior_gap}/{total_tracks}")
print()
print("-- Pass 2 --")
print(f"stress-tested: {len(ids_to_test)}   collisions found: {n_collision}   "
      f"robustly safe: {n_safe}")
print(f"non-vehicle challengers: {n_non_vehicle}/{len(challenger_types)}")
print()
print("-- Pass 3 (export_shard_geometry — first execution ever) --")
print(f"exported: {summary['exported']}   agents_written: {summary['agents_written']}   "
      f"agents_skipped: {summary['agents_skipped']}   errors: {len(summary['errors'])}")
print(f"geometry M-ordinate verification: "
      f"{'PASSED' if n_exact_match == n_checked else 'FAILED'} "
      f"({n_exact_match}/{n_checked} agents)")
print()
print("-- Wall clock --")
print(f"Pass 1: {pass1_elapsed:.1f}s   Pass 2: {pass2_elapsed:.1f}s   "
      f"Pass 3: {pass3_elapsed:.1f}s")
print(f"total notebook compute (excludes installs/mount): "
      f"{pass1_elapsed + pass2_elapsed + pass3_elapsed:.1f}s")